# Sarcasm detection — Colab training (RoBERTa-base)

Self-contained notebook for Helal et al. (2024), *A contextual-based approach for sarcasm detection*.

You do **not** need the local `sarc` folder. This notebook downloads the News Headlines dataset, trains **RoBERTa-base**, and lets you download the model zip.

## How to run on Google Colab

1. Open [Google Colab](https://colab.research.google.com/)
2. **File → Upload notebook** and choose `colab_train.ipynb`
3. **Runtime → Change runtime type → Hardware accelerator → T4 GPU** (the paper used a T4)
4. **Runtime → Run all**

When training finishes, download `roberta_headline_only.zip`. That zip is the model you will load later on your PC.


## 1. Install packages and check the GPU


In [ ]:
# Colab already has CUDA PyTorch when you pick a GPU runtime.
# Do not `pip install torch` here — that can replace the GPU build with CPU.
!pip -q install -U transformers scikit-learn seaborn tqdm

import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime → Change runtime type → T4 GPU, then re-run this cell.")
    print("CPU training on the full dataset can take many hours.")


## 2. Training config

Paper Table 4 defaults: batch 32, lr `5e-5`, 5 epochs, eval batch 64, weight decay 0.01.

Leave `MAX_SAMPLES = None` for the full ~26.7k headlines. Set it to `2000` only for a quick test.


In [ ]:
from pathlib import Path

IN_COLAB = Path("/content").exists()
WORK_DIR = Path("/content/sarc_run") if IN_COLAB else Path.cwd() / "sarc_run"
DATA_DIR = WORK_DIR / "data"
MODEL_DIR = WORK_DIR / "models" / "roberta_headline_only"
RESULTS_DIR = WORK_DIR / "results"
for path in (DATA_DIR, MODEL_DIR, RESULTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "roberta-base"
MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 5e-5
EPOCHS = 5
WEIGHT_DECAY = 0.01
SEED = 42
MAX_SAMPLES = None          # None = full dataset (paper setting)
CONTEXT_TYPE = "headline_only"
SAVE_TO_DRIVE = False       # set True after mounting Drive if you want a backup

print("WORK_DIR:", WORK_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("EPOCHS:", EPOCHS, "| MAX_SAMPLES:", MAX_SAMPLES)


## 3. Optional: save a copy to Google Drive

Skip this cell unless you want the model kept after Colab disconnects.


In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted. The model will be copied there after training.")
else:
    print("Drive backup is off. You can still download the zip at the end.")


## 4. Download the News Headlines dataset


In [ ]:
import json
import urllib.request
import pandas as pd

RAW_PATH = DATA_DIR / "Sarcasm_Headlines_Dataset.json"
DATASET_URLS = [
    "https://huggingface.co/datasets/AgamP/sarcasm-detection/resolve/main/Sarcasm_Headlines_Dataset.json",
    "https://raw.githubusercontent.com/rishabhmisra/News-Headlines-Dataset-For-Sarcasm-Detection/master/Sarcasm_Headlines_Dataset.json",
]

def download_dataset(dest: Path) -> Path:
    if dest.exists() and dest.stat().st_size > 1000:
        print("Already downloaded:", dest)
        return dest
    last_error = None
    for url in DATASET_URLS:
        try:
            print("Downloading:", url)
            urllib.request.urlretrieve(url, dest)
            print("Saved:", dest, "bytes:", dest.stat().st_size)
            return dest
        except Exception as exc:
            last_error = exc
            print("Failed:", url, "->", exc)
    raise RuntimeError(
        "Could not download the dataset. Upload Sarcasm_Headlines_Dataset.json into "
        f"{DATA_DIR} and re-run this cell."
    ) from last_error

download_dataset(RAW_PATH)

rows = []
with RAW_PATH.open(encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)
need = {"headline", "is_sarcastic"}
missing = need - set(df.columns)
if missing:
    raise ValueError(f"Dataset missing columns: {sorted(missing)}")
if "author" not in df.columns:
    df["author"] = "unknown"
if "section" not in df.columns:
    df["section"] = "unknown"
if "description" not in df.columns:
    df["description"] = "unknown"
df = df.drop_duplicates(subset=["headline", "is_sarcastic"]).reset_index(drop=True)

print("Rows:", len(df))
print("Columns:", list(df.columns))
print(df["is_sarcastic"].value_counts())
df.head(3)


## 5. Class distribution


In [ ]:
import matplotlib.pyplot as plt

counts = df["is_sarcastic"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Non-sarcastic (0)", "Sarcastic (1)"], [counts.get(0, 0), counts.get(1, 0)],
       color=["#4C78A8", "#F58518"])
ax.set_title("News Headlines class distribution")
ax.set_ylabel("Count")
fig.tight_layout()
plt.show()


## 6. Helper functions

Input format matches the paper idea:

```
HEADLINE: <headline>
```

Later you can switch `CONTEXT_TYPE` to `all_context` if you add author/section/description.


In [ ]:
import html
import random
import re
import time
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, RobertaForSequenceClassification

WHITESPACE_RE = re.compile(r"\s+")
CONTEXT_TYPES = ("headline_only", "author", "section", "description", "all_context")


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def clean_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "unknown"
    text = html.unescape(str(value)).replace("\xa0", " ").strip()
    text = WHITESPACE_RE.sub(" ", text)
    return text if text else "unknown"


def build_input_text(row, context_type: str = "headline_only") -> str:
    if context_type not in CONTEXT_TYPES:
        raise ValueError(f"Unknown context_type={context_type}")
    lines = [f"HEADLINE: {clean_text(row.get('headline'))}"]
    if context_type in ("author", "all_context"):
        lines.append(f"AUTHOR: {clean_text(row.get('author'))}")
    if context_type in ("section", "all_context"):
        lines.append(f"SECTION: {clean_text(row.get('section'))}")
    if context_type in ("description", "all_context"):
        lines.append(f"DESCRIPTION: {clean_text(row.get('description'))}")
    return "\n".join(lines)


def maybe_sample(frame: pd.DataFrame, max_samples: Optional[int]) -> pd.DataFrame:
    if max_samples is None or max_samples >= len(frame):
        return frame.reset_index(drop=True)
    sampled, _ = train_test_split(
        frame, train_size=max_samples, random_state=SEED, stratify=frame["is_sarcastic"]
    )
    return sampled.reset_index(drop=True)


def stratified_splits(frame: pd.DataFrame):
    # 80/20 train/test, then 12.5% of train as validation (~70/10/20)
    train_full, test_df = train_test_split(
        frame, test_size=0.20, random_state=SEED, stratify=frame["is_sarcastic"]
    )
    train_df, val_df = train_test_split(
        train_full, test_size=0.125, random_state=SEED, stratify=train_full["is_sarcastic"]
    )
    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


class SarcasmDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: torch.tensor(value[index]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(int(self.labels[index]), dtype=torch.long)
        return item


def compute_metrics(y_true, y_pred) -> Dict[str, float]:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


print("Helpers ready.")


## 7. Build inputs, split, and tokenize


In [ ]:
set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

work = maybe_sample(df, MAX_SAMPLES).copy()
work["input_text"] = [build_input_text(row, CONTEXT_TYPE) for row in work.to_dict("records")]
train_df, val_df, test_df = stratified_splits(work)

print("Dataset size:", len(work))
print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))
print("Example input:")
print(work.loc[0, "input_text"])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_column(texts):
    return tokenizer(
        list(texts),
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

lengths = [
    len(tokenizer(text, truncation=False, add_special_tokens=True)["input_ids"])
    for text in work["input_text"]
]
print("Token length min/mean/max:", min(lengths), round(sum(lengths)/len(lengths), 2), max(lengths))
print("Share over 256:", round(sum(l > 256 for l in lengths) / len(lengths), 4))

train_ds = SarcasmDataset(tokenize_column(train_df["input_text"]), train_df["is_sarcastic"].to_numpy())
val_ds = SarcasmDataset(tokenize_column(val_df["input_text"]), val_df["is_sarcastic"].to_numpy())
test_ds = SarcasmDataset(tokenize_column(test_df["input_text"]), test_df["is_sarcastic"].to_numpy())

train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False)


## 8. Train RoBERTa-base

This is the long cell. On a Colab T4, 5 epochs over the full dataset often takes about 30–90 minutes. Keep this tab open so Colab does not disconnect.


In [ ]:
model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
use_amp = device == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


def run_epoch(loader, train_mode: bool):
    model.train(train_mode)
    losses, y_true, y_pred = [], [], []
    context = torch.enable_grad() if train_mode else torch.no_grad()
    with context:
        for batch in tqdm(loader, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            if train_mode:
                optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                # labels → CrossEntropyLoss inside RobertaForSequenceClassification
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
            if train_mode:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            losses.append(float(loss.item()))
            preds = torch.argmax(outputs.logits, dim=-1)
            y_true.extend(labels.detach().cpu().tolist())
            y_pred.extend(preds.detach().cpu().tolist())
    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(np.mean(losses))
    return metrics


history = []
best_f1 = -1.0
best_state = None
t0 = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    print()
    print(f"Epoch {epoch}/{EPOCHS}")
    train_metrics = run_epoch(train_loader, train_mode=True)
    val_metrics = run_epoch(val_loader, train_mode=False)
    history.append({
        "epoch": epoch,
        "loss": train_metrics["loss"],
        "eval_loss": val_metrics["loss"],
        "eval_accuracy": val_metrics["accuracy"],
        "eval_f1": val_metrics["f1"],
    })
    print(
        f"  train_loss={train_metrics['loss']:.4f}  val_loss={val_metrics['loss']:.4f}  "
        f"val_acc={val_metrics['accuracy']:.4f}  val_f1={val_metrics['f1']:.4f}"
    )
    if val_metrics["f1"] > best_f1:
        best_f1 = val_metrics["f1"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print("  saved new best model by validation F1")

print("Training seconds:", round(time.perf_counter() - t0, 1))
if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)


## 9. Test evaluation


In [ ]:
model.eval()
all_logits, all_labels, test_losses = [], [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="test"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        test_losses.append(float(outputs.loss.item()))
        all_logits.append(outputs.logits.detach().cpu())
        all_labels.extend(labels.detach().cpu().tolist())

logits = torch.cat(all_logits, dim=0)
probs = torch.softmax(logits, dim=-1)[:, 1].numpy()
y_pred = logits.argmax(dim=-1).numpy()
y_true = np.asarray(all_labels)
test_metrics = compute_metrics(y_true, y_pred)
test_metrics["test_loss"] = float(np.mean(test_losses))

print("Test metrics (ours, not copied from the paper):")
for key, value in test_metrics.items():
    print(f"  {key}: {value:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=["Non-sarcastic", "Sarcastic"], digits=4))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-sarcastic", "Sarcastic"],
            yticklabels=["Non-sarcastic", "Sarcastic"], ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix (headline only)")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "confusion_matrix_headline_only.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([h["epoch"] for h in history], [h["loss"] for h in history], marker="o", label="Train loss")
ax.plot([h["epoch"] for h in history], [h["eval_loss"] for h in history], marker="o", label="Val loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training loss vs validation loss")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "loss_curve.png", dpi=150)
plt.show()

pd.DataFrame(history)


## 10. Save the model and download the zip


In [ ]:
import shutil

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
meta = {
    "context_type": CONTEXT_TYPE,
    "base_model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "test_metrics": test_metrics,
    "label_map": {"0": "non_sarcastic", "1": "sarcastic"},
}
(MODEL_DIR / "export_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

zip_path = shutil.make_archive(str(WORK_DIR / "roberta_headline_only"), "zip", MODEL_DIR)
print("Model folder:", MODEL_DIR)
print("Zip:", zip_path)
print("Files:", sorted(p.name for p in MODEL_DIR.iterdir()))

if SAVE_TO_DRIVE:
    drive_dir = Path("/content/drive/MyDrive/sarc_models/roberta_headline_only")
    if drive_dir.exists():
        shutil.rmtree(drive_dir)
    shutil.copytree(MODEL_DIR, drive_dir)
    print("Copied to Drive:", drive_dir)

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
    print("Download started: roberta_headline_only.zip")
else:
    print("Not in Colab — zip is on disk at", zip_path)


## 11. Quick inference test

After you download the zip, unzip it on your PC to a folder such as `models/roberta_headline_only` and use:

```python
from transformers import AutoTokenizer, RobertaForSequenceClassification
import torch

path = "models/roberta_headline_only"
tokenizer = AutoTokenizer.from_pretrained(path)
model = RobertaForSequenceClassification.from_pretrained(path)
model.eval()

text = "HEADLINE: stock analysts confused, frightened by boar market"
enc = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=256)
with torch.no_grad():
    prob = torch.softmax(model(**enc).logits, dim=-1)[0, 1].item()
print("Sarcastic" if prob >= 0.5 else "Non-sarcastic", f"{prob*100:.2f}%")
```


In [ ]:
def predict_headline(headline: str) -> None:
    text = build_input_text({"headline": headline}, "headline_only")
    enc = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=MAX_LENGTH)
    enc = {k: v.to(device) for k, v in enc.items()}
    model.eval()
    with torch.no_grad():
        prob = torch.softmax(model(**enc).logits, dim=-1)[0, 1].item()
    label = "Sarcastic" if prob >= 0.5 else "Non-sarcastic"
    print(text)
    print(f"Prediction: {label}")
    print(f"Sarcasm probability: {prob * 100:.2f}%")
    print()

predict_headline("stock analysts confused, frightened by boar market")
predict_headline("obama visits arlington national cemetery to honor veterans")


## Notes

- Report **this notebook's** accuracy / F1. Do not copy the paper's 99.7% unless you actually get it.
- This run is headline-only. Author/section/description are `unknown` unless you later add scraped context.
- Keep the whole unzipped folder. You need `config.json`, tokenizer files, and the weights, not a single `.bin` by itself.
